In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = [15, 6]
%config InlineBackend.figure_format = 'retina'

import plotly.express as px
import plotly.graph_objects as go

/kaggle/input/police-deaths-in-usa-from-1791-to-2022/scrapper.py
/kaggle/input/police-deaths-in-usa-from-1791-to-2022/police_deaths_in_america.csv


# Dataset: Police Deaths in USA from 1791 to 2022
### Analyzed on June 9, 2022

The dataset contains 10 features of which 8 are of string type and 1 is Boolean while another being integer. This dataset contains the information about the police officers who have passed away while on duty. These are the data points collected and stored by the Officer Down Memorial Page Organization. Let us perform EDA & Time Series Forcasting on the dataset to understand the relationship between various features present in the dataset.

In [2]:
df = pd.read_csv('/kaggle/input/police-deaths-in-usa-from-1791-to-2022/police_deaths_in_america.csv')
df

,Rank,Name,Cause_of_Death,Date,Year,Month,Day,Department,State,K9_Unit
0,Constable,Darius Quimby,Stabbed,"Monday, January 3, 1791",1791,January,Monday,"Albany County Constable's Office, NY",New York,0
1,Sheriff,Cornelius Hogeboom,Gunfire,"Saturday, October 22, 1791",1791,October,Saturday,"Columbia County Sheriff's Office, NY",New York,0
2,Deputy Sheriff,Isaac Smith,Gunfire,"Thursday, May 17, 1792",1792,May,Thursday,"Westchester County Sheriff's Department, NY",New York,0
3,Marshal,Robert Forsyth,Gunfire,"Saturday, January 11, 1794",1794,January,Saturday,United States Department of Justice - United S...,United States,0
4,Deputy Sheriff,Robert Berwick,Gunfire,"Thursday, June 29, 1797",1797,June,Thursday,"New York County Sheriff's Office, NY",New York,0
...,...,...,...,...,...,...,...,...,...,...
26264,K9,Ciro,Fire,"Thursday, March 3, 2022",2022,March,Thursday,"Humphreys County Sheriff's Office, TN",Tennessee,1
26265,K9,Dash,Gunfire,"Wednesday, March 9, 2022",2022,March,Wednesday,"Shepherdsville Police Department, KY",Kentucky,1
26266,K9,Major,Gunfire,"Sunday, April 10, 2022",2022,April,Sunday,"Franklin County Sheriff's Office, NC",North Carolina,1
26267,K9,Jinx,Gunfire,"Monday, April 11, 2022",2022,April,Monday,"El Paso County Sheriff's Office, CO",Colorado,1


In [3]:
px.line(pd.DataFrame(df.groupby('Year').count().drop(['Name','Cause_of_Death','Date','State','Month','Day','Department','K9_Unit'],axis=1).rename(columns={'Rank':'Number of Deaths'})),title='Deaths')

On examining the above file, we see a presence of a boolean value. The k9 unit's deployment and mode of operation is different to that of human police force. Therefore both of the sets of data are to be processed differently.

# Human Police Force

In [4]:
k9_df = df[df['K9_Unit'] == 1].reset_index(drop=True)
df = df[df['K9_Unit'] == 0].reset_index(drop=True)
df.drop('K9_Unit',axis=1,inplace=True)
k9_df.drop('K9_Unit',axis=1,inplace=True) # Since the split is done based on the K9_Unit column, it is no longer required for further analysis.
k9_df

,Rank,Name,Cause_of_Death,Date,Year,Month,Day,Department,State
0,K9,Vag,Struck by streetcar,"Thursday, April 12, 1877",1877,April,Thursday,"Buffalo Police Department, NY",New York
1,K9,Danger,Struck by train,"Tuesday, June 9, 1896",1896,June,Tuesday,"New York, Chicago and St. Louis Railroad Polic...",United States
2,K9,Ollie,Struck by vehicle,"Thursday, August 26, 1909",1909,August,Thursday,"New York City Police Department, NY",New York
3,K9,Trooper,Assault,"Friday, March 17, 1961",1961,March,Friday,"Kansas City Police Department, MO",Missouri
4,K9,Baron,Assault,"Wednesday, September 12, 1962",1962,September,Wednesday,"Virginia State Police, VA",Virginia
...,...,...,...,...,...,...,...,...,...
478,K9,Ciro,Fire,"Thursday, March 3, 2022",2022,March,Thursday,"Humphreys County Sheriff's Office, TN",Tennessee
479,K9,Dash,Gunfire,"Wednesday, March 9, 2022",2022,March,Wednesday,"Shepherdsville Police Department, KY",Kentucky
480,K9,Major,Gunfire,"Sunday, April 10, 2022",2022,April,Sunday,"Franklin County Sheriff's Office, NC",North Carolina
481,K9,Jinx,Gunfire,"Monday, April 11, 2022",2022,April,Monday,"El Paso County Sheriff's Office, CO",Colorado


In [5]:
df

,Rank,Name,Cause_of_Death,Date,Year,Month,Day,Department,State
0,Constable,Darius Quimby,Stabbed,"Monday, January 3, 1791",1791,January,Monday,"Albany County Constable's Office, NY",New York
1,Sheriff,Cornelius Hogeboom,Gunfire,"Saturday, October 22, 1791",1791,October,Saturday,"Columbia County Sheriff's Office, NY",New York
2,Deputy Sheriff,Isaac Smith,Gunfire,"Thursday, May 17, 1792",1792,May,Thursday,"Westchester County Sheriff's Department, NY",New York
3,Marshal,Robert Forsyth,Gunfire,"Saturday, January 11, 1794",1794,January,Saturday,United States Department of Justice - United S...,United States
4,Deputy Sheriff,Robert Berwick,Gunfire,"Thursday, June 29, 1797",1797,June,Thursday,"New York County Sheriff's Office, NY",New York
...,...,...,...,...,...,...,...,...,...
25781,Supervisory Police Officer,"Yiu Tak ""Louis"" Tao",9/11 related illness,"Tuesday, May 17, 2022",2022,May,Tuesday,United States Department of Justice - Federal ...,United States
25782,Senior Correctional Police Officer,Daniel Sincavage,Automobile crash,"Thursday, May 19, 2022",2022,May,Thursday,"New Jersey Department of Corrections, NJ",New Jersey
25783,Officer Trainee,Cody Alan Olafson,Duty related illness,"Friday, May 20, 2022",2022,May,Friday,United States Department of Homeland Security ...,United States
25784,Police Officer,Houston Tipping,Training accident,"Sunday, May 29, 2022",2022,May,Sunday,"Los Angeles Police Department, CA",California


#### There are 25786 instances of police officers falling down in Non-K9 units while there are 483 instances of officers in K9 units that have fallen down.

## Non K9 Officer data analysis

### State-wise analysis

In [6]:
df_state = df.groupby('State').count().drop(['Name','Cause_of_Death','Date','Year','Month','Day','Department'],axis=1)
df_state = df_state.rename(columns={'Rank':'Deaths'}).sort_values(by='Deaths',ascending=False)
df_state.head(10)

,Deaths
State,
Texas,2203
United States,1951
New York,1868
California,1730
Illinois,1158
Pennsylvania,1086
Kentucky,938
Florida,924
Ohio,888


In [7]:
df_state.drop('United States',axis=0,inplace=True) # We remove this category as we are interested to know the state department statistics rather than federal office statistics.
fig = px.bar(df_state,title='Officers fallen down by state')
fig.update_traces(textfont_size=12, textangle=45, textposition="outside",cliponaxis=False)
fig.show()

We can observe from the above plot that the **Texas**, **New York** and **California** have far more deaths than the next state ,**Illinois**. So let us analyze other statistics about these states in detail.

In [8]:
df_interested_states = df[df['State'].isin(['California','Texas','New York'])]
df_interested_states

,Rank,Name,Cause_of_Death,Date,Year,Month,Day,Department,State
0,Constable,Darius Quimby,Stabbed,"Monday, January 3, 1791",1791,January,Monday,"Albany County Constable's Office, NY",New York
1,Sheriff,Cornelius Hogeboom,Gunfire,"Saturday, October 22, 1791",1791,October,Saturday,"Columbia County Sheriff's Office, NY",New York
2,Deputy Sheriff,Isaac Smith,Gunfire,"Thursday, May 17, 1792",1792,May,Thursday,"Westchester County Sheriff's Department, NY",New York
4,Deputy Sheriff,Robert Berwick,Gunfire,"Thursday, June 29, 1797",1797,June,Thursday,"New York County Sheriff's Office, NY",New York
8,Watchman,Christian Luswanger,Stabbed,"Thursday, December 25, 1806",1806,December,Thursday,"New York City Night Watch, NY",New York
...,...,...,...,...,...,...,...,...,...
25758,Sergeant,Barbara Majors Fenley,Fire,"Thursday, March 17, 2022",2022,March,Thursday,"Eastland County Sheriff's Office, TX",Texas
25767,Deputy Sheriff,Darren Almendarez,Gunfire,"Thursday, March 31, 2022",2022,March,Thursday,"Harris County Sheriff's Office, TX",Texas
25770,Deputy Constable,Jennifer Lauren Chavis,Vehicular assault,"Saturday, April 2, 2022",2022,April,Saturday,"Harris County Constable's Office - Precinct 7, TX",Texas
25778,Deputy Sheriff,Robert Adam Howard,Automobile crash,"Wednesday, May 11, 2022",2022,May,Wednesday,"Harris County Sheriff's Office, TX",Texas


In [9]:
df_interested_states = pd.DataFrame(df_interested_states.groupby(by=['Year','State'],as_index=False).count().drop(['Name','Cause_of_Death','Date','Month','Day','Department'],axis=1))
df_interested_states = df_interested_states.rename(columns={'Rank':'Deaths'})
df_interested_states

,Year,State,Deaths
0,1791,New York,2
1,1792,New York,1
2,1797,New York,1
3,1806,New York,1
4,1818,New York,1
...,...,...,...
503,2021,New York,28
504,2021,Texas,107
505,2022,California,7
506,2022,New York,4


In [10]:
fig2 = px.line(df_interested_states,y='Deaths',x='Year',color='State',markers=True)
fig2.update_layout(hovermode="x unified")
fig2.show()

Based on the above graph, we can see that new york has the oldest records of the 3 states. The earlier records are absent as the states were not added to the constitution. Texas was added to the USA around the years of 1830 - 1845. California was added to the Constitution of USA in 1850. Prior to these years, the data is not present and even if present need not obey the same rules of recording the relevant statistics. 

In [11]:
fig2 = px.line(df_interested_states[df_interested_states['Year'] > 1980],y='Deaths',x='Year',color='State',markers=True,title='Deaths from 1980')
fig2.update_layout(hovermode="x unified")
fig2.show()

In [12]:
df_ny = df[(df['State'] == 'New York') & (df['Year'] == 2001)].groupby('Cause_of_Death').count().drop(['Name','State','Date','Year','Month','Day','Department'],axis=1)
df_ny.rename(columns ={'Rank':'Count'})
df_ny

,Rank
Cause_of_Death,
Terrorist attack,69


In [13]:
df_tx_2021 = df[(df['State'] == 'Texas') & (df['Year'] == 2021)].groupby('Cause_of_Death').count().drop(['Name','State','Date','Year','Month','Day','Department'],axis=1)
df_tx_2021.rename(columns ={'Rank':'Count'})
df_tx_2021

,Rank
Cause_of_Death,
Automobile crash,1
COVID19,95
Gunfire,8
Heart attack,2
Vehicular assault,1


In [14]:
df_tx_2020 = df[(df['State'] == 'Texas') & (df['Year'] == 2020)].groupby('Cause_of_Death').count().drop(['Name','State','Date','Year','Month','Day','Department'],axis=1)
df_tx_2020.rename(columns ={'Rank':'Count'})
df_tx_2020

,Rank
Cause_of_Death,
Aircraft accident,1
Automobile crash,1
COVID19,59
Gunfire,7
Gunfire (Inadvertent),1
Heart attack,2
Heatstroke,1
Struck by vehicle,2
Vehicular assault,2


On observing the data from recent past (~ 40 years), we see two spikes in deaths. One among them is the 9/11 attack and all the police deaths recorded in New York that year are from the terrorist attack. While the bigger peak in texas corresponds majority of the deaths occuring from Covid-19 pandemic. 

### Day-wise analysis

In [15]:
px.bar(df.groupby('Day').count().drop(['Name','State','Date','Year','Month','Rank','Department'],axis=1).rename(columns={'Cause_of_Death':'Count'}),title='Deaths on various days')

From the above bar chart we can see that Saturday has the highest deaths while Wednesday has the least number of deaths.

### Cause of Death Analysis

In [16]:
px.bar(df.groupby('Cause_of_Death').count().drop(['Name','State','Date','Year','Month','Rank','Department'],axis=1).rename(columns={'Day':'Count'}),title='Deaths due to various reasons')

From the above graph, we can look at the cause of death. The cause that is the most popular is Gunfire. The second most popular is Automobile Crash. These reasons are so ubiqutous that no secondary analysis are required into the dataset for this feature. 

### Rank wise Analysis

In [17]:
df_rank = df.groupby('Rank').count().drop(['Name','State','Date','Year','Month','Cause_of_Death','Department'],axis=1).rename(columns={'Day':'Count'})
px.bar(pd.DataFrame(df_rank[df_rank['Count']>100]),title='Deaths due to various reasons')

**Patrolman**, **Police Officer** and **Deputy Sheriff**s are the most dangerous positions/ranks in the police departments. Those in these positions need additional support force to better protect their lives. Some of the less important death rate among the ranks are ignored to show the important ones better in the graph.

## Forecasting

Developing some models to predict the number of police deaths for the next year. Using the algorith developed in [this](https://arxiv.org/abs/1905.10437) paper, we can perform univariate time series modeling. The features present in the dataset do not necessarily explain or cause the deaths of the police officers, therefore we cannot perform multi variate time series forecasting. We can perform feature engineering by combining data from other datasets and events that happened in the same time frame to create better models. 

In [18]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers
tf.__version__

'2.6.4'

In [19]:
# Creating NBeats block

class NBeatsBlock(Layer):
    def __init__(self,
               input_size: int,
               theta_size: int,
               horizon: int,
               n_neurons: int,
               n_layers: int,
               **kwargs):
        super().__init__(**kwargs)
        self.input_size = input_size
        self.theta_size = theta_size
        self.horizon = horizon
        self.n_neurons = n_neurons
        self.n_layers = n_layers
        
        # Block contains stack of 4 fully connected layers each has ReLU activation
        self.hidden = Sequential([])
        for _ in range(n_layers):
            self.hidden.add(Dense(n_neurons,activation='relu'))
        # Output of block is a theta layer with linear activation
        self.theta_layer = tf.keras.layers.Dense(theta_size, activation="linear", name="theta")
    
    def call(self,inputs):
        theta = self.theta_layer(self.hidden(inputs))
        
        backcast, forecast = theta[:,:self.input_size], theta[:,-self.horizon:]
        
        return backcast, forecast


dataset = pd.DataFrame(df.groupby('Year',as_index=False).count().drop(['Name','Cause_of_Death','Date','State','Month','Day','Department'],axis=1)).rename(columns={'Rank':'Count'})
dataset = dataset[dataset['Year'] > 1850]
dataset

,Year,Count
38,1851,9
39,1852,9
40,1853,9
41,1854,10
42,1855,15
...,...,...
205,2018,188
206,2019,158
207,2020,414
208,2021,622


In [20]:
from tensorflow.keras.utils import timeseries_dataset_from_array

HORIZON = 1
WINDOW_SIZE = 15

inputs = dataset['Count'][:-WINDOW_SIZE]
targets = dataset['Count'][WINDOW_SIZE:]

split_size = int(len(dataset) * 0.8)
X_train, y_train = inputs[:split_size], targets[:split_size]
X_test, y_test = inputs[split_size:], targets[split_size:]
len(X_train), len(y_train), len(X_test), len(y_test)

trainset = timeseries_dataset_from_array(X_train, y_train, sequence_length=WINDOW_SIZE,batch_size=1024).prefetch(tf.data.AUTOTUNE)
testset = timeseries_dataset_from_array(X_test, y_test, sequence_length=WINDOW_SIZE,batch_size=1024).prefetch(tf.data.AUTOTUNE)

trainset, testset

2022-06-11 03:31:22.384947: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-06-11 03:31:22.389425: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-06-11 03:31:22.390122: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-06-11 03:31:22.392075: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compil

(<PrefetchDataset shapes: ((None, None), (None,)), types: (tf.int64, tf.int64)>,
 <PrefetchDataset shapes: ((None, None), (None,)), types: (tf.int64, tf.int64)>)

In [21]:
N_EPOCHS = 500 # called "Iterations" in Table 18
N_NEURONS = 512 # called "Width" in Table 18
N_LAYERS = 4
N_STACKS = 30

INPUT_SIZE = WINDOW_SIZE * HORIZON # called "Lookback" in Table 18
THETA_SIZE = INPUT_SIZE + HORIZON

INPUT_SIZE, THETA_SIZE

(15, 16)

In [22]:
# 1. Setup N-BEATS Block layer
nbeats_block_layer = NBeatsBlock(input_size=INPUT_SIZE,
                                 theta_size=THETA_SIZE,
                                 horizon=HORIZON,
                                 n_neurons=N_NEURONS,
                                 n_layers=N_LAYERS,
                                 name="InitialBlock")

# 2. Create input to stacks
stack_input = layers.Input(shape=(INPUT_SIZE), name="stack_input")

# 3. Create initial backcast and forecast input (backwards predictions are referred to as residuals in the paper)
backcast, forecast = nbeats_block_layer(stack_input)
# Add in subtraction residual link, thank you to: https://github.com/mrdbourke/tensorflow-deep-learning/discussions/174 
residuals = layers.subtract([stack_input, backcast], name=f"subtract_00") 

# 4. Create stacks of blocks
for i, _ in enumerate(range(N_STACKS-1)): # first stack is already creted in (3)

  # 5. Use the NBeatsBlock to calculate the backcast as well as block forecast
  backcast, block_forecast = NBeatsBlock(
      input_size=INPUT_SIZE,
      theta_size=THETA_SIZE,
      horizon=HORIZON,
      n_neurons=N_NEURONS,
      n_layers=N_LAYERS,
      name=f"NBeatsBlock_{i}"
  )(residuals) # pass it in residuals (the backcast)

  # 6. Create the double residual stacking
  residuals = layers.subtract([residuals, backcast], name=f"subtract_{i}") 
  forecast = layers.add([forecast, block_forecast], name=f"add_{i}")

# 7. Put the stack model together
model_nbeats = tf.keras.Model(inputs=stack_input, 
                         outputs=forecast, 
                         name="model_7_N-BEATS")

# 8. Compile with MAE loss and Adam optimizer
model_nbeats.compile(loss="mae",
                optimizer=tf.keras.optimizers.Adam(0.001),
                metrics=["mae", "mse"])

# 9. Fit the model with EarlyStopping and ReduceLROnPlateau callbacks
model_nbeats.fit(trainset,
            epochs=N_EPOCHS,
            validation_data=testset,
            verbose=1,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=200, restore_best_weights=True),
                      tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=100, verbose=1)])

2022-06-11 03:31:26.592636: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/500
1/1 [==============================] - 9s 9s/step - loss: 132.1064 - mae: 132.1064 - mse: 22262.6680 - val_loss: 7315.0464 - val_mae: 7315.0464 - val_mse: 53527212.0000
Epoch 2/500
1/1 [==============================] - 0s 104ms/step - loss: 6056.0479 - mae: 6056.0479 - mse: 48805380.0000 - val_loss: 1168.1207 - val_mae: 1168.1207 - val_mse: 1368868.0000
Epoch 3/500
1/1 [==============================] - 0s 108ms/step - loss: 970.0338 - mae: 970.0338 - mse: 1310381.2500 - val_loss: 429.4260 - val_mae: 429.4260 - val_mse: 184555.0469
Epoch 4/500
1/1 [==============================] - 0s 105ms/step - loss: 350.0040 - mae: 350.0040 - mse: 177683.6406 - val_loss: 421.6145 - val_mae: 421.6145 - val_mse: 194644.3594
Epoch 5/500
1/1 [==============================] - 0s 105ms/step - loss: 395.1009 - mae: 395.1009 - mse: 217646.7656 - val_loss: 16.1685 - val_mae: 16.1685 - val_mse: 355.3372
Epoch 6/500
1/1 [==============================] - 0s 107ms/step - loss: 35.5499 - mae: 35.5

In [23]:
model_nbeats.evaluate(testset)

1/1 [==============================] - 0s 69ms/step - loss: 11.7399 - mae: 11.7399 - mse: 289.3073


[11.73986530303955, 11.73986530303955, 289.3072509765625]

In [24]:
model_nbeats.predict(tf.expand_dims(dataset['Count'][-WINDOW_SIZE:],axis=0))

array([[140.4808]], dtype=float32)

Provided that there is no strong policy change and change in behaviour of the people over 2023, it is expected that there would be **140 police officers** will be sacrifying their lives in the year 2023 with most of them being Patrolman, Police Officer and Deputy Sheriffs.

## K9 Analysis

In [25]:
px.bar(k9_df.groupby('Cause_of_Death').count().drop(['Name','Date','Year','Month','Day','Department','State'],axis=1).rename(columns={'Rank':'Count'}).sort_values(by='Count',ascending=False))

In [26]:
px.bar(k9_df.groupby('State').count().drop(['Name','Date','Year','Month','Day','Department','Cause_of_Death'],axis=1).rename(columns={'Rank':'Count'}).sort_values(by='Count',ascending=False))

In [27]:
k9_list = []
iter_year = 1950
rows = pd.DataFrame(k9_df[k9_df['Year'] > 1961].groupby('Year',as_index=False).count()).drop(['Rank','Name','Date','Month','Day','Department','State'],axis=1).rename(columns={'Cause_of_Death':'Count'}).values.tolist()
i = 0
while True:
    if rows[i][0] == iter_year:
        k9_list.append(rows[i][1])
        i = i + 1
    else:
        k9_list.append(0)
    iter_year = iter_year + 1
    if iter_year > max([i[0] for i in rows]):
        break
k9_list

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 2,
 0,
 1,
 1,
 1,
 1,
 3,
 0,
 4,
 4,
 3,
 2,
 3,
 6,
 2,
 3,
 4,
 2,
 5,
 5,
 6,
 7,
 5,
 3,
 3,
 3,
 0,
 4,
 4,
 4,
 6,
 6,
 7,
 8,
 9,
 6,
 10,
 7,
 6,
 9,
 3,
 9,
 10,
 14,
 7,
 14,
 18,
 22,
 20,
 27,
 36,
 24,
 28,
 28,
 22,
 21,
 9]

In [28]:
HORIZON = 1
WINDOW_SIZE = 7

inputs = k9_list[:-WINDOW_SIZE]
targets = k9_list[WINDOW_SIZE:]

split_size = int(len(k9_list) * 0.8)
X_train, y_train = inputs[:split_size], targets[:split_size]
X_test, y_test = inputs[split_size:], targets[split_size:]

trainset = timeseries_dataset_from_array(X_train, y_train, sequence_length=WINDOW_SIZE,batch_size=1024).prefetch(tf.data.AUTOTUNE)
testset = timeseries_dataset_from_array(X_test, y_test, sequence_length=WINDOW_SIZE,batch_size=1024).prefetch(tf.data.AUTOTUNE)

trainset, testset

(<PrefetchDataset shapes: ((None, None), (None,)), types: (tf.int32, tf.int32)>,
 <PrefetchDataset shapes: ((None, None), (None,)), types: (tf.int32, tf.int32)>)

In [29]:
INPUT_SIZE = WINDOW_SIZE * HORIZON # called "Lookback" in Table 18
THETA_SIZE = INPUT_SIZE + HORIZON

# 1. Setup N-BEATS Block layer
nbeats_block_layer = NBeatsBlock(input_size=INPUT_SIZE,
                                 theta_size=THETA_SIZE,
                                 horizon=HORIZON,
                                 n_neurons=N_NEURONS,
                                 n_layers=N_LAYERS,
                                 name="InitialBlock")

# 2. Create input to stacks
stack_input = layers.Input(shape=(INPUT_SIZE), name="stack_input")

# 3. Create initial backcast and forecast input (backwards predictions are referred to as residuals in the paper)
backcast, forecast = nbeats_block_layer(stack_input)
# Add in subtraction residual link, thank you to: https://github.com/mrdbourke/tensorflow-deep-learning/discussions/174 
residuals = layers.subtract([stack_input, backcast], name=f"subtract_00") 

# 4. Create stacks of blocks
for i, _ in enumerate(range(N_STACKS-1)): # first stack is already creted in (3)

  # 5. Use the NBeatsBlock to calculate the backcast as well as block forecast
  backcast, block_forecast = NBeatsBlock(
      input_size=INPUT_SIZE,
      theta_size=THETA_SIZE,
      horizon=HORIZON,
      n_neurons=N_NEURONS,
      n_layers=N_LAYERS,
      name=f"NBeatsBlock_{i}"
  )(residuals) # pass it in residuals (the backcast)

  # 6. Create the double residual stacking
  residuals = layers.subtract([residuals, backcast], name=f"subtract_{i}") 
  forecast = layers.add([forecast, block_forecast], name=f"add_{i}")

# 7. Put the stack model together
model_nbeats = tf.keras.Model(inputs=stack_input, 
                         outputs=forecast, 
                         name="model_7_N-BEATS")

# 8. Compile with MAE loss and Adam optimizer
model_nbeats.compile(loss="mae",
                optimizer=tf.keras.optimizers.Adam(0.001),
                metrics=["mae", "mse"])

# 9. Fit the model with EarlyStopping and ReduceLROnPlateau callbacks
model_nbeats.fit(trainset,
            epochs=N_EPOCHS,
            validation_data=testset,
            verbose=1,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=200, restore_best_weights=True),
                      tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=100, verbose=1)])

Epoch 1/500
1/1 [==============================] - 8s 8s/step - loss: 4.1110 - mae: 4.1110 - mse: 27.2578 - val_loss: 275.1852 - val_mae: 275.1852 - val_mse: 76139.5391
Epoch 2/500
1/1 [==============================] - 0s 92ms/step - loss: 53.3505 - mae: 53.3505 - mse: 4316.3838 - val_loss: 49.8197 - val_mae: 49.8197 - val_mse: 2484.3989
Epoch 3/500
1/1 [==============================] - 0s 92ms/step - loss: 22.5719 - mae: 22.5719 - mse: 757.2276 - val_loss: 23.7254 - val_mae: 23.7254 - val_mse: 563.0187
Epoch 4/500
1/1 [==============================] - 0s 95ms/step - loss: 8.0478 - mae: 8.0478 - mse: 85.8783 - val_loss: 11.3742 - val_mae: 11.3742 - val_mse: 138.6433
Epoch 5/500
1/1 [==============================] - 0s 95ms/step - loss: 1.5661 - mae: 1.5661 - mse: 4.6962 - val_loss: 30.2953 - val_mae: 30.2953 - val_mse: 931.6605
Epoch 6/500
1/1 [==============================] - 0s 97ms/step - loss: 3.9245 - mae: 3.9245 - mse: 22.3735 - val_loss: 20.6062 - val_mae: 20.6062 - val_mse

In [30]:
model_nbeats.predict(tf.expand_dims(k9_list[-WINDOW_SIZE:],axis=0))

array([[26.017815]], dtype=float32)

Based on the same prediction model, there would be **26 K9 deaths** in the next year 2023 provided all the contributing factors continue to influence in the same way. Most of the dogs would like be passed due to **gunfire** and the most risk is involved in **California** and **Florida.**

### Thank you for going through my notebook. Please leave a comment about what can be done better for this notebook.